# End-to-End Customer Churn ML Pipeline with Scikit-learn Pipeline API

Welcome to the self-contained Jupyter Notebook designed to build, optimize, and export a production-ready machine learning pipeline to predict customer churn using the **IBM Telco Churn Dataset**.

### Objectives of this notebook:
1. **Clean & Parse Schema**: Handle empty spaces in numerical features and map columns properly.
2. **Construct scikit-learn Pipeline**: Chain preprocessing and estimator fitting into a single robust step.
3. **Evaluate Multiple Models via GridSearchCV**: Run a joint search comparing **Logistic Regression** and **Random Forest Classifier**.
4. **Serialize the Entire Pipeline**: Save the finalized pipeline using `joblib` so that it can be served instantly in a web app (e.g. Streamlit).

In [ ]:
# Step 1: Install required scientific dependencies
!pip install -q scikit-learn joblib pandas numpy matplotlib seaborn

In [ ]:
# Step 2: System Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import joblib

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, f1_score, roc_curve, precision_recall_curve

print("✔ Libraries imported successfully.")

### Step 3: First Principles Preprocessing Design

We define our column groups and custom cleaning logic. 
* **Numerical features** (`tenure`, `MonthlyCharges`, `TotalCharges`): Standardized and imputed with median values.
* **Categorical features**: One-hot encoded and imputed with the most frequent value.
* **TotalCharges gotcha**: Empty string spaces (`" "`) are parsed to `NaN` using `pd.to_numeric` with `errors='coerce'`.

In [ ]:
NUMERICAL_COLS = ["tenure", "MonthlyCharges", "TotalCharges"]

CATEGORICAL_COLS = [
    "gender", "SeniorCitizen", "Partner", "Dependents", 
    "PhoneService", "MultipleLines", "InternetService", 
    "OnlineSecurity", "OnlineBackup", "DeviceProtection", 
    "TechSupport", "StreamingTV", "StreamingMovies", 
    "Contract", "PaperlessBilling", "PaymentMethod"
]

def preprocess_raw_data(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    
    # Drop identifier if present
    if "customerID" in df.columns:
        df = df.drop(columns=["customerID"])
        
    # Coerce TotalCharges to numeric
    if "TotalCharges" in df.columns:
        df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
        
    # Clean SeniorCitizen representation
    if "SeniorCitizen" in df.columns:
        df["SeniorCitizen"] = df["SeniorCitizen"].map({0: "No", 1: "Yes"}).fillna("No")
        
    return df

def build_preprocessor() -> ColumnTransformer:
    numerical_transformer = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ])

    categorical_transformer = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
    ])

    preprocessor = ColumnTransformer(
        transformers=[
            ("num", numerical_transformer, NUMERICAL_COLS),
            ("cat", categorical_transformer, CATEGORICAL_COLS)
        ],
        remainder="drop"
    )
    return preprocessor

print("✔ Modular preprocessing pipelines designed.")

### Step 4: Load Churn Dataset from GitHub

We pull the official raw IBM dataset directly from GitHub.

In [ ]:
DATA_URL = "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv"
raw_df = pd.read_csv(DATA_URL)
print(f"Raw dataset loaded. Total records: {len(raw_df)}")
raw_df.head(3)

### Step 5: Clean and Train-Test Split

Apply our cleaning function and perform a stratified train-test split to preserve target labels.

In [ ]:
X = raw_df.drop(columns=["Churn"])
y = raw_df["Churn"].map({"No": 0, "Yes": 1})

X_cleaned = preprocess_raw_data(X)

# Stratified Train-Test split
X_train, X_test, y_train, y_test = train_test_split(
    X_cleaned, y, 
    test_size=0.2, 
    random_state=42, 
    stratify=y
)

print(f"Training set shape: {X_train.shape}")
print(f"Test set shape    : {X_test.shape}")

### Step 6: Define Pipeline & Multi-Model GridSearchCV Search

We build the end-to-end `Pipeline` and define a parameter grid comparing Logistic Regression and Random Forest models.

In [ ]:
preprocessor = build_preprocessor()

main_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(max_iter=1000))
])

param_grid = [
    {
        "classifier": [LogisticRegression(max_iter=1000, random_state=42)],
        "classifier__C": [0.01, 0.1, 1.0, 10.0],
        "classifier__solver": ["liblinear", "lbfgs"]
    },
    {
        "classifier": [RandomForestClassifier(random_state=42)],
        "classifier__n_estimators": [100, 200],
        "classifier__max_depth": [5, 10, 15, None],
        "classifier__min_samples_split": [2, 5]
    }
    ]

print("Starting Hyperparameter Optimization Grid Search...")
grid_search = GridSearchCV(
    estimator=main_pipeline,
    param_grid=param_grid,
    scoring="f1",
    cv=5,
    n_jobs=-1,
    verbose=1
)

grid_search.fit(X_train, y_train)
print("✔ Grid Search completed successfully.")

### Step 7: Evaluate Model Performance

Extract best parameters and evaluate model predictions against classification metrics, ROC curve, and confusion matrices.

In [ ]:
best_pipeline = grid_search.best_estimator_
print(f"Best Model   : {best_pipeline.named_steps['classifier']}")
print(f"Best Params  : {grid_search.best_params_}\n")

y_pred = best_pipeline.predict(X_test)
y_proba = best_pipeline.predict_proba(X_test)[:, 1]

print("--- Test Set Classification Report ---")
print(classification_report(y_test, y_pred, target_names=["No Churn", "Churn"], digits=4))

roc_auc = roc_auc_score(y_test, y_proba)
print(f"ROC-AUC Score    : {roc_auc:.4%}")
print(f"Test Set F1-Score: {f1_score(y_test, y_pred):.4%}")

# Plot Confusion Matrix
plt.figure(figsize=(5, 4))
sns.heatmap(confusion_matrix(y_test, y_pred), annot=True, fmt="d", cmap="Blues", cbar=False,
            xticklabels=["No Churn", "Churn"], yticklabels=["No Churn", "Churn"])
plt.title("Confusion Matrix")
plt.ylabel("True Class")
plt.xlabel("Predicted Class")
plt.show()

### Step 8: Plot ROC Curve & Precision-Recall Curve

Visualize model performance characteristics.

In [ ]:
fpr, tpr, _ = roc_curve(y_test, y_proba)
precision, recall, _ = precision_recall_curve(y_test, y_proba)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# ROC Curve
ax1.plot(fpr, tpr, color="darkorange", lw=2, label=f"ROC Curve (AUC = {roc_auc:.4f})")
ax1.plot([0, 1], [0, 1], color="navy", lw=2, linestyle="--")
ax1.set_xlim([0.0, 1.0])
ax1.set_ylim([0.0, 1.05])
ax1.set_xlabel("False Positive Rate")
ax1.set_ylabel("True Positive Rate")
ax1.set_title("Receiver Operating Characteristic (ROC)")
ax1.legend(loc="lower right")

# Precision-Recall Curve
ax2.plot(recall, precision, color="blue", lw=2, label="PR Curve")
ax2.set_xlabel("Recall")
ax2.set_ylabel("Precision")
ax2.set_title("Precision-Recall Curve")
ax2.legend(loc="lower left")

plt.tight_layout()
plt.show()

### Step 9: Serialize the Complete Pipeline

Save the end-to-end best pipeline to a single production file.

In [ ]:
MODEL_FILENAME = "customer_churn_model.joblib"
print(f"Saving finalized pipeline to: {MODEL_FILENAME}")
joblib.dump(best_pipeline, MODEL_FILENAME)
print("🎉 SUCCESS! customer_churn_model.joblib has been exported successfully.")

### Step 10: Validation Test (Simulate Web Serving API)

Load the serialized model back and pass a single raw dictionary representing a single customer. This verifies that our ColumnTransformer correctly scales numerical inputs and handles one-hot encoding on the fly without any manual transformations!

In [ ]:
loaded_pipeline = joblib.load(MODEL_FILENAME)

# Define mock customer details
mock_customer = {
    "gender": ["Female"],
    "SeniorCitizen": [0],
    "Partner": ["No"],
    "Dependents": ["No"],
    "tenure": [3],
    "PhoneService": ["Yes"],
    "MultipleLines": ["No"],
    "InternetService": ["Fiber optic"],
    "OnlineSecurity": ["No"],
    "OnlineBackup": ["No"],
    "DeviceProtection": ["No"],
    "TechSupport": ["No"],
    "StreamingTV": ["No"],
    "StreamingMovies": ["No"],
    "Contract": ["Month-to-month"],
    "PaperlessBilling": ["Yes"],
    "PaymentMethod": ["Electronic check"],
    "MonthlyCharges": [70.50],
    "TotalCharges": [211.50]
}

mock_df = pd.DataFrame(mock_customer)
prob = loaded_pipeline.predict_proba(mock_df)[0][1]
pred = loaded_pipeline.predict(mock_df)[0]

print(f"\nMock Prediction Successful!")
print(f"Predicted Churn Probability: {prob:.2%}")
print(f"Predicted Churn Label       : {'Yes' if pred == 1 else 'No'}")